In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Phase 2 — Classification Preprocessing Pipeline (PyTorch)

This notebook validates the image preprocessing pipeline for the aerial object classification task.

## Goals
- load configs and project paths
- scan the processed classification dataset
- create deterministic class mappings
- build PyTorch datasets and dataloaders
- verify image shape and normalization
- generate preview figures

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/Aerial_Object_Classification_Detection")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.utils.config import load_yaml
from src.utils.logger import get_logger
from src.utils.paths import load_paths_config, ensure_dir
from src.utils.seed import set_seed
from src.utils.plotting import save_tensor_preview
from src.utils.io import save_text

from src.data.classification_loader import build_classification_loader_pipeline
from src.features.class_mapping import save_class_mapping_artifact
from src.features.preprocessing_transforms import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    build_pytorch_transfer_transform,
)
from src.data.preprocess_classification import build_resize_only_transform

ModuleNotFoundError: No module named 'tensorflow'

## Load configs and initialize paths
This cell loads:
- `paths.yaml`
- `classification_config.yaml`
- `transfer_learning_config.yaml`

It also initializes logging and reproducibility.

In [3]:
configs_dir = PROJECT_ROOT / "configs"

paths_config = load_yaml(configs_dir / "paths.yaml")
classification_config = load_yaml(configs_dir / "classification_config.yaml")
transfer_learning_config = load_yaml(configs_dir / "transfer_learning_config.yaml")

path_map = load_paths_config(configs_dir / "paths.yaml")

classification_root = path_map.get("processed_classification_root", PROJECT_ROOT / "data" / "processed" / "classification")
figures_preprocessing_dir = ensure_dir(PROJECT_ROOT / "figures" / "preprocessing")
reports_dir = ensure_dir(PROJECT_ROOT / "reports")
interim_previews_dir = ensure_dir(PROJECT_ROOT / "data" / "interim" / "previews")
logs_dir = ensure_dir(PROJECT_ROOT / "logs")

logger = get_logger(
    name="phase2_preprocessing_pipeline",
    log_file=logs_dir / "phase2_preprocessing_pipeline.log",
)

seed = transfer_learning_config["training"]["seed"]
set_seed(seed)

logger.info("Configs loaded successfully.")
logger.info("Phase 2 preprocessing pipeline started.")
print("Classification dataset root:", classification_root)
print("Figures dir:", figures_preprocessing_dir)

2026-03-29 08:37:44 | INFO | phase2_preprocessing_pipeline | Configs loaded successfully.
2026-03-29 08:37:44 | INFO | phase2_preprocessing_pipeline | Phase 2 preprocessing pipeline started.


Classification dataset root: /content/drive/MyDrive/Aerial_Object_Classification_Detection/data/processed/classification
Figures dir: /content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/preprocessing


## Build the classification loader pipeline
This cell validates the processed classification dataset structure and builds:
- train dataset
- validation dataset
- test dataset
- train/validation/test dataloaders
- class mappings

In [6]:
artifacts = build_classification_loader_pipeline(
    dataset_root=classification_root,
    classification_config=classification_config,
    transfer_learning_config=transfer_learning_config,
    logger=logger,
)

scan_df = artifacts["scan_df"]
class_to_index = artifacts["class_to_index"]
index_to_class = artifacts["index_to_class"]
dataloaders = artifacts["dataloaders"]

print("Class to index:", class_to_index)
print("Index to class:", index_to_class)
display(scan_df)

2026-03-29 08:55:15 | INFO | phase2_preprocessing_pipeline | Validating classification dataset structure...
2026-03-29 08:55:16 | INFO | phase2_preprocessing_pipeline | Building PyTorch classification datasets...
2026-03-29 08:55:17 | INFO | phase2_preprocessing_pipeline | Building PyTorch dataloaders...
2026-03-29 08:55:17 | INFO | phase2_preprocessing_pipeline | Classification loader pipeline built successfully.


Class to index: {'bird': 0, 'drone': 1}
Index to class: {0: 'bird', 1: 'drone'}


,split,class_name,image_count,class_dir_exists
0,train,bird,1414,True
1,train,drone,1248,True
2,valid,bird,217,True
3,valid,drone,225,True
4,test,bird,121,True
5,test,drone,94,True


## Save class mapping artifact
This cell saves a reusable JSON artifact containing:
- class-to-index mapping
- index-to-class mapping

In [7]:
mapping_file_name = classification_config["dataset"]["class_mapping"]["mapping_file_name"]
mapping_output_path = interim_previews_dir / mapping_file_name

save_class_mapping_artifact(
    class_to_index=class_to_index,
    index_to_class=index_to_class,
    output_path=mapping_output_path,
)

print("Saved mapping artifact to:", mapping_output_path)

Saved mapping artifact to: /content/drive/MyDrive/Aerial_Object_Classification_Detection/data/interim/previews/class_mapping.json


## Validate loader output shapes
This cell proves that:
- data loads correctly
- tensor batch shapes are correct
- labels are integer encoded correctly
- standard preprocessing is applied

In [12]:
train_images, train_labels = next(iter(dataloaders["train"]))
valid_images, valid_labels = next(iter(dataloaders["valid"]))
test_images, test_labels = next(iter(dataloaders["test"]))

print("Train batch image shape :", tuple(train_images.shape))
print("Train batch label shape :", tuple(train_labels.shape))
print("Valid batch image shape :", tuple(valid_images.shape))
print("Test batch image shape  :", tuple(test_images.shape))

print("Train tensor min/max:", float(train_images.min()), float(train_images.max()))
print("Valid tensor min/max:", float(valid_images.min()), float(valid_images.max()))
print("Test tensor min/max :", float(test_images.min()), float(test_images.max()))

assert train_images.ndim == 4
assert train_images.shape[1] == 3
assert train_images.shape[2] == 224
assert train_images.shape[3] == 224

assert float(train_images.min()) >= 0.0
assert float(train_images.max()) <= 1.0

assert set(class_to_index.keys()) == set(classification_config["dataset"]["expected_classes"])
assert set(index_to_class.values()) == set(classification_config["dataset"]["expected_classes"])

print("All basic pipeline checks passed.")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Train batch image shape : (32, 3, 224, 224)
Train batch label shape : (32,)
Valid batch image shape : (32, 3, 224, 224)
Test batch image shape  : (32, 3, 224, 224)
Train tensor min/max: 0.0 1.0
Valid tensor min/max: 0.0 1.0
Test tensor min/max : 0.0 1.0
All basic pipeline checks passed.


## Save resized preview
This preview shows the post-resize image shape before any ImageNet transfer normalization branch is applied.

In [13]:
first_train_dataset = artifacts["datasets"]["train"]
first_image_tensor, first_label = first_train_dataset[0]

resized_preview_path = figures_preprocessing_dir / "resized_preview.png"

save_tensor_preview(
    tensor=first_image_tensor,
    output_path=resized_preview_path,
    title=f"Resized Preview | Label: {index_to_class[first_label]}",
)

print("Saved resized preview to:", resized_preview_path)

Saved resized preview to: /content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/preprocessing/resized_preview.png


## Validate transfer-learning preprocessing hook
This cell applies the PyTorch transfer-learning branch:
- resize
- tensor conversion
- ImageNet normalization

The saved preview is denormalized for visualization only.

In [14]:
from PIL import Image

sample_path, sample_label_idx = first_train_dataset.samples[0]
sample_pil = Image.open(sample_path).convert("RGB")

transfer_transform = build_pytorch_transfer_transform(image_size=(224, 224))
transfer_tensor = transfer_transform(sample_pil)

print("Transfer tensor shape:", tuple(transfer_tensor.shape))
print("Transfer tensor min/max:", float(transfer_tensor.min()), float(transfer_tensor.max()))

Transfer tensor shape: (3, 224, 224)
Transfer tensor min/max: -2.032280206680298 2.640000104904175


In [15]:
normalized_preview_path = figures_preprocessing_dir / "normalized_preview.png"

save_tensor_preview(
    tensor=transfer_tensor,
    output_path=normalized_preview_path,
    title=f"Transfer-Normalized Preview | Label: {index_to_class[sample_label_idx]}",
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
)

print("Saved normalized preview to:", normalized_preview_path)

Saved normalized preview to: /content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/preprocessing/normalized_preview.png


## Generate notebook summary for report
This cell creates a markdown summary that can be pasted into `reports/preprocessing_report.md` if needed.

In [16]:
split_summary = scan_df.groupby("split", as_index=False)["image_count"].sum()

summary_lines = [
    "# Phase 2 Preprocessing Validation Summary",
    "",
    "## Dataset root",
    f"- {classification_root}",
    "",
    "## Class mappings",
    f"- class_to_index: {class_to_index}",
    f"- index_to_class: {index_to_class}",
    "",
    "## Split counts",
    "",
    split_summary.to_markdown(index=False),
    "",
    "## Batch checks",
    f"- Train batch shape: {tuple(train_images.shape)}",
    f"- Valid batch shape: {tuple(valid_images.shape)}",
    f"- Test batch shape: {tuple(test_images.shape)}",
    f"- Train tensor min/max: ({float(train_images.min()):.4f}, {float(train_images.max()):.4f})",
    "",
    "## Saved figures",
    f"- {resized_preview_path}",
    f"- {normalized_preview_path}",
]

summary_text = "\n".join(summary_lines)
summary_output_path = reports_dir / "preprocessing_report.md"
save_text(summary_text, summary_output_path)

print(summary_text)
print("\nSaved preprocessing report to:", summary_output_path)

# Phase 2 Preprocessing Validation Summary

## Dataset root
- /content/drive/MyDrive/Aerial_Object_Classification_Detection/data/processed/classification

## Class mappings
- class_to_index: {'bird': 0, 'drone': 1}
- index_to_class: {0: 'bird', 1: 'drone'}

## Split counts

| split   |   image_count |
|:--------|--------------:|
| test    |           215 |
| train   |          2662 |
| valid   |           442 |

## Batch checks
- Train batch shape: (32, 3, 224, 224)
- Valid batch shape: (32, 3, 224, 224)
- Test batch shape: (32, 3, 224, 224)
- Train tensor min/max: (0.0000, 1.0000)

## Saved figures
- /content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/preprocessing/resized_preview.png
- /content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/preprocessing/normalized_preview.png

Saved preprocessing report to: /content/drive/MyDrive/Aerial_Object_Classification_Detection/reports/preprocessing_report.md
